# 9-10 - Training-Only Augmentation and Augmentation Audit - All Families

This notebook implements:

**Step 09 — Augment the inner-training observations only**

Target post-augmentation class distributions:

- 50/50
- 80/20
- 85/15
- 90/10

The first value is the proportion of `y=0` non-interactions and the second value is the proportion of `y=1` interactions.

**Step 10 — Audit augmentation**

The fixed train/validation/test roles created previously are treated as immutable. Only observations with `role == "train"` are available to any augmentation, resampling, cleaning, or generative procedure. Validation and outer-test observations are never used for fitting scalers, nearest-neighbor models, Gaussian mixtures, VAE models, WGAN-GP models, or class-ratio adjustment.

The 13 augmentation algorithms represented here are:

1. Random Over-Sampling
2. Random Under-Sampling
3. Tomek Links
4. CNN + Tomek Links
5. SMOTE
6. SMOTE + Tomek Links
7. Basic SMOTE
8. ADASYN
9. k-Means SMOTE
10. Latent Sampling
11. Noise Injection
12. VAE
13. WGAN-GP
14. Latent Diffusion

The first 11 are evaluated on the MSCMF representation. VAE and WGAN-GP are evaluated on MSCMF, NNMF, and LMF. Latent Diffusion is evaluated on the MSCMF representation. This produces 18 method–backbone configurations per family/fold.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

try:
    from imblearn.over_sampling import (
        SMOTE,
        ADASYN,
        KMeansSMOTE,
    )
    from imblearn.under_sampling import (
        TomekLinks,
        CondensedNearestNeighbour,
    )
except ImportError as exc:
    raise ImportError(
        "This notebook requires imbalanced-learn. "
        "Install it with: pip install imbalanced-learn"
    ) from exc

try:
    import tensorflow as tf
    import keras
    from tensorflow.keras import layers, models, backend as K
    from tensorflow.keras.optimizers import Adam
except ImportError as exc:
    raise ImportError(
        "This notebook requires TensorFlow/Keras."
    ) from exc

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "split"
STATS_DIR = PROJECT_ROOT / "Stats"

STATS_DIR.mkdir(parents=True, exist_ok=True)

assert SPLIT_DIR.exists(), f"Split folder not found: {SPLIT_DIR}"

print("Project root :", PROJECT_ROOT)
print("Split folder :", SPLIT_DIR)
print("Stats folder :", STATS_DIR)

C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\riskf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Pyt

Project root : c:\Users\riskf\OneDrive\A-DTI2026
Split folder : c:\Users\riskf\OneDrive\A-DTI2026\data\split
Stats folder : c:\Users\riskf\OneDrive\A-DTI2026\Stats


## 1. Fixed configuration

No data split is generated in this notebook.

Random state `42` is retained where it was explicit in the supplied classical resampling implementations. VAE, WGAN-GP, and Gaussian-noise generation remain stochastic where the source implementation did not define a fixed TensorFlow/NumPy training seed.

In [2]:
FAMILIES = {
    "enzyme": {"display": "Enzyme"},
    "gpcr": {"display": "GPCR"},
    "ion_channel": {"display": "Ion Channel"},
    "nuclear_receptor": {"display": "Nuclear Receptor"},
}

BACKBONES = ["MSCMF", "NNMF", "LMF"]

CLASSICAL_MSCMF_METHODS = [
    "RandomOver",
    "RandomUnder",
    "TomekLinks",
    "CNNTomekLinks",
    "SMOTE",
    "SMOTETomekLinks",
    "BasicSMOTE",
    "ADASYN",
    "KMeansSMOTE",
    "LatentSampling",
    "NoiseInjection",
]

GENERATIVE_METHODS = [
    "VAE",
    "WGANGP",
]

DIFFUSION_METHODS = [
    "LatentDiffusion",
]

METHOD_BACKBONE_CONFIGS = (
    [("MSCMF", method) for method in CLASSICAL_MSCMF_METHODS]
    + [
        (backbone, method)
        for backbone in BACKBONES
        for method in GENERATIVE_METHODS
    ]
    + [("MSCMF", "LatentDiffusion")]
)

assert len(CLASSICAL_MSCMF_METHODS) == 11
assert len(
    set(
        CLASSICAL_MSCMF_METHODS
        + GENERATIVE_METHODS
        + DIFFUSION_METHODS
    )
) == 14
assert len(METHOD_BACKBONE_CONFIGS) == 18

# Exact non-interaction : interaction integer ratios.
TARGET_RATIOS = {
    "50_50": (1, 1),
    "80_20": (4, 1),
    "85_15": (17, 3),
    "90_10": (9, 1),
}

N_OUTER_FOLDS = 5
RANDOM_STATE = 42

FEATURE_DIM = 100

# VAE source parameters
VAE_LATENT_DIM = 16
VAE_EPOCHS = 4
VAE_BATCH_SIZE = 77

# WGAN-GP source parameters
WGAN_NOISE_DIM = 100
WGAN_EPOCHS = 100
WGAN_BATCH_SIZE = 256
WGAN_LEARNING_RATE = 0.0001
WGAN_BETA_1 = 0.5
WGAN_GP_LAMBDA = 10.0

# Noise-injection source parameter
NOISE_SIGMA = 0.01

# Latent Diffusion source parameters
DIFFUSION_VAE_LATENT_DIM = 32
DIFFUSION_VAE_EPOCHS = 10
DIFFUSION_VAE_LEARNING_RATE = 0.001
DIFFUSION_VAE_BETA = 0.1

DIFFUSION_EPOCHS = 50
DIFFUSION_TIMESTEPS = 1000
DIFFUSION_SAMPLING_STEPS = 100
DIFFUSION_LEARNING_RATE = 1e-4
DIFFUSION_BETA_START = 0.0001
DIFFUSION_BETA_END = 0.02

IMPLEMENTATION_VERSION = "augmentation_v2_train_only_14_methods_with_diffusion"

RESUME_IF_VALID = True

print("Unique augmentation algorithms:", 14)
print("Method-backbone configurations:", len(METHOD_BACKBONE_CONFIGS))
print("Implementation version:", IMPLEMENTATION_VERSION)

Unique augmentation algorithms: 14
Method-backbone configurations: 18
Implementation version: augmentation_v2_train_only_14_methods_with_diffusion


## 2. Utilities for fixed Step 06 roles and Step 07-08 latent embeddings

In [3]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def row_col_columns(df):
    if {"y_row", "y_col"}.issubset(df.columns):
        return "y_row", "y_col"
    if {"protein_index", "compound_index"}.issubset(df.columns):
        return "protein_index", "compound_index"
    raise AssertionError(
        "Role file must contain y_row/y_col or "
        "protein_index/compound_index."
    )


def latent_dir(family, outer_fold, backbone):
    return (
        SPLIT_DIR
        / family
        / "latent"
        / f"outer_fold_{outer_fold}"
        / backbone.lower()
    )


def augmentation_dir(family, outer_fold, backbone, method):
    return (
        SPLIT_DIR
        / family
        / "augmentation"
        / f"outer_fold_{outer_fold}"
        / backbone.lower()
        / method
    )


def load_training_representation(family, outer_fold, backbone):
    role_file = (
        SPLIT_DIR
        / family
        / f"outer_fold_{outer_fold}_roles.csv"
    )
    assert role_file.exists(), f"Missing role file: {role_file}"

    roles_all = pd.read_csv(role_file)
    assert roles_all["pair_id"].is_unique
    assert set(roles_all["role"].unique()) == {
        "train", "validation", "test"
    }

    train_roles = (
        roles_all.loc[roles_all["role"].eq("train")]
        .copy()
        .reset_index(drop=True)
    )

    row_col, col_col = row_col_columns(train_roles)

    p_idx = train_roles[row_col].astype(int).to_numpy()
    c_idx = train_roles[col_col].astype(int).to_numpy()

    ldir = latent_dir(family, outer_fold, backbone)
    protein_file = ldir / "protein_embeddings.npy"
    compound_file = ldir / "compound_embeddings.npy"
    metadata_file = ldir / "fit_metadata.json"

    for p in [protein_file, compound_file, metadata_file]:
        assert p.exists(), f"Missing Step 07-08 output: {p}"

    A = np.load(protein_file).astype(np.float32)
    B = np.load(compound_file).astype(np.float32)

    assert A.shape[1] == 50
    assert B.shape[1] == 50

    # Step 07-08 convention:
    # A = protein embeddings, B = compound embeddings.
    X = np.concatenate(
        [A[p_idx], B[c_idx]],
        axis=1,
    ).astype(np.float32)

    y = train_roles["y"].astype(np.int8).to_numpy()

    assert X.shape == (len(train_roles), FEATURE_DIM)
    assert set(np.unique(y)).issubset({0, 1})
    assert np.isfinite(X).all()

    return {
        "X": X,
        "y": y,
        "train_roles": train_roles,
        "roles_all": roles_all,
        "role_file": role_file,
        "role_file_sha256": file_sha256(role_file),
        "latent_metadata_file": metadata_file,
        "latent_metadata_sha256": file_sha256(metadata_file),
    }

## 3. Exact-ratio planner



In [4]:
def exact_ratio_plan(
    y_real,
    real_allowed_indices,
    n_synthetic_positive,
    ratio_name,
    random_state=RANDOM_STATE,
):
    neg_units, pos_units = TARGET_RATIOS[ratio_name]

    real_allowed_indices = np.asarray(
        real_allowed_indices,
        dtype=np.int64,
    )

    y_allowed = y_real[real_allowed_indices]

    neg_real = real_allowed_indices[y_allowed == 0]
    pos_real = real_allowed_indices[y_allowed == 1]

    n0_available = len(neg_real)
    n1_available = len(pos_real) + int(n_synthetic_positive)

    k = min(
        n0_available // neg_units,
        n1_available // pos_units,
    )

    if k < 1:
        raise ValueError(
            f"Not enough candidate observations for ratio {ratio_name}."
        )

    target_n0 = neg_units * k
    target_n1 = pos_units * k

    rng = np.random.RandomState(random_state)

    selected_neg = (
        neg_real
        if len(neg_real) == target_n0
        else rng.choice(
            neg_real,
            size=target_n0,
            replace=False,
        )
    )

    # Prefer real positive observations. Synthetic positives are added
    # only when the requested target exceeds the available real positives.
    n_real_pos_take = min(
        len(pos_real),
        target_n1,
    )

    selected_pos_real = (
        pos_real
        if len(pos_real) == n_real_pos_take
        else rng.choice(
            pos_real,
            size=n_real_pos_take,
            replace=False,
        )
    )

    n_syn_take = target_n1 - n_real_pos_take

    if n_syn_take > n_synthetic_positive:
        raise AssertionError(
            "Requested more synthetic positives than available."
        )

    selected_syn = np.arange(
        n_syn_take,
        dtype=np.int64,
    )

    selected_real = np.concatenate(
        [selected_neg, selected_pos_real]
    ).astype(np.int64)

    y_plan = np.concatenate([
        y_real[selected_real],
        np.ones(n_syn_take, dtype=np.int8),
    ])

    n0 = int((y_plan == 0).sum())
    n1 = int((y_plan == 1).sum())

    assert n0 * pos_units == n1 * neg_units

    return {
        "real_indices": selected_real,
        "synthetic_indices": selected_syn,
        "n0": n0,
        "n1": n1,
        "n_total": n0 + n1,
        "n_real": len(selected_real),
        "n_synthetic": n_syn_take,
    }


def target_positive_count_for_max_pool(y):
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())

    # 50/50 is the most demanding positive proportion.
    return max(0, n0 - n1)

# Step 09 — Augmentation implementations

## 4. Classical and latent-space methods


In [5]:
def extract_synthetic_tail(
    X_resampled,
    y_resampled,
    n_original,
):
    X_resampled = np.asarray(X_resampled)
    y_resampled = np.asarray(y_resampled)

    if len(X_resampled) < n_original:
        raise AssertionError(
            "Oversampler returned fewer rows than the original training data."
        )

    synthetic_X = X_resampled[n_original:]
    synthetic_y = y_resampled[n_original:]

    if len(synthetic_y):
        assert np.all(synthetic_y == 1)

    return synthetic_X.astype(np.float32)


def pool_random_over(X, y):
    pos = X[y == 1]
    n_syn = target_positive_count_for_max_pool(y)

    if n_syn == 0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    synthetic = resample(
        pos,
        replace=True,
        n_samples=n_syn,
        random_state=RANDOM_STATE,
    )

    return np.asarray(synthetic, dtype=np.float32)


def pool_smote_scaled(X, y):
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    if n1 >= n0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    sm = SMOTE(
        sampling_strategy={1: n0},
        random_state=RANDOM_STATE,
    )
    X_res, y_res = sm.fit_resample(X_scaled, y)

    synthetic_scaled = extract_synthetic_tail(
        X_res,
        y_res,
        len(X),
    )

    return scaler.inverse_transform(
        synthetic_scaled
    ).astype(np.float32)


def pool_basic_smote(X, y):
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    if n1 >= n0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    sm = SMOTE(
        sampling_strategy={1: n0},
        random_state=RANDOM_STATE,
    )
    X_res, y_res = sm.fit_resample(X, y)

    return extract_synthetic_tail(
        X_res,
        y_res,
        len(X),
    )


def pool_adasyn(X, y):
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    if n1 >= n0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    ada = ADASYN(
        sampling_strategy={1: n0},
        random_state=RANDOM_STATE,
    )
    X_res, y_res = ada.fit_resample(X, y)

    return extract_synthetic_tail(
        X_res,
        y_res,
        len(X),
    )


def pool_kmeans_smote(X, y):
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    if n1 >= n0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    kmeans_model = KMeans(
        n_clusters=5,
        random_state=RANDOM_STATE,
    )

    sampler = KMeansSMOTE(
        sampling_strategy={1: n0},
        cluster_balance_threshold=0.01,
        kmeans_estimator=kmeans_model,
        random_state=RANDOM_STATE,
    )

    X_res, y_res = sampler.fit_resample(X, y)

    return extract_synthetic_tail(
        X_res,
        y_res,
        len(X),
    )


def pool_latent_sampling(X, y):
    positive = X[y == 1]
    n_syn = target_positive_count_for_max_pool(y)

    if n_syn == 0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    gmm = GaussianMixture(
        n_components=1,
        covariance_type="full",
        random_state=RANDOM_STATE,
    )
    gmm.fit(positive)

    synthetic = gmm.sample(
        n_samples=n_syn
    )[0]

    return synthetic.astype(np.float32)


def pool_noise_injection(X, y):
    positive = X[y == 1]
    n_syn = target_positive_count_for_max_pool(y)

    if n_syn == 0:
        return np.empty((0, X.shape[1]), dtype=np.float32)

    # The supplied implementation does not define a fixed seed for
    # np.random.normal. Preserve that stochastic behavior.
    source_indices = np.random.choice(
        len(positive),
        size=n_syn,
        replace=True,
    )

    base = positive[source_indices]

    noise = np.random.normal(
        loc=0.0,
        scale=NOISE_SIGMA,
        size=base.shape,
    )

    return (base + noise).astype(np.float32)


def real_allowed_tomek(X, y):
    tl = TomekLinks()
    tl.fit_resample(X, y)

    return np.asarray(
        tl.sample_indices_,
        dtype=np.int64,
    )


def real_allowed_cnn_tomek(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    cnn = CondensedNearestNeighbour(
        n_neighbors=1,
        random_state=RANDOM_STATE,
    )
    X_cnn, y_cnn = cnn.fit_resample(
        X_scaled,
        y,
    )

    cnn_indices = np.asarray(
        cnn.sample_indices_,
        dtype=np.int64,
    )

    tl = TomekLinks()
    tl.fit_resample(
        X_cnn,
        y_cnn,
    )

    tomek_on_cnn = np.asarray(
        tl.sample_indices_,
        dtype=np.int64,
    )

    return cnn_indices[tomek_on_cnn]


def smote_tomek_pool(X, y):
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())

    if n1 >= n0:
        return (
            np.arange(len(X), dtype=np.int64),
            np.empty((0, X.shape[1]), dtype=np.float32),
        )

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    sm = SMOTE(
        sampling_strategy={1: n0},
        random_state=RANDOM_STATE,
    )
    X_smote, y_smote = sm.fit_resample(
        X_scaled,
        y,
    )

    tl = TomekLinks()
    X_clean, y_clean = tl.fit_resample(
        X_smote,
        y_smote,
    )

    kept = np.asarray(
        tl.sample_indices_,
        dtype=np.int64,
    )

    # SMOTE appends generated observations after the original rows.
    real_mask = kept < len(X)
    real_allowed = kept[real_mask]

    synthetic_positions = kept[~real_mask] - len(X)
    synthetic_scaled_all = X_smote[len(X):]

    synthetic_kept_scaled = (
        synthetic_scaled_all[synthetic_positions]
        if len(synthetic_positions)
        else np.empty(
            (0, X.shape[1]),
            dtype=np.float64,
        )
    )

    synthetic_kept = scaler.inverse_transform(
        synthetic_kept_scaled
    ).astype(np.float32)

    return (
        real_allowed.astype(np.int64),
        synthetic_kept,
    )

## 5. VAE augmentation


In [6]:
# ============================================================
# VAE AUGMENTATION
# ============================================================


class Sampling(
    layers.Layer
):

    def call(
        self,
        inputs,
    ):

        (
            z_mean,
            z_log_var,
        ) = inputs

        batch = tf.shape(
            z_mean
        )[0]

        dim = tf.shape(
            z_mean
        )[1]

        epsilon = (
            K.random_normal(
                shape=(
                    batch,
                    dim,
                )
            )
        )

        return (
            z_mean
            + tf.exp(
                0.5
                * z_log_var
            )
            * epsilon
        )


# ============================================================
# Encoder
# ============================================================

def build_vae_encoder(
    latent_dim=VAE_LATENT_DIM,
):

    encoder_inputs = (
        layers.Input(
            shape=(
                FEATURE_DIM,
            )
        )
    )

    x = layers.Dense(
        256,
        activation="relu",
    )(
        encoder_inputs
    )

    x = layers.Dense(
        64,
        activation="relu",
    )(
        x
    )

    z_mean = layers.Dense(
        latent_dim
    )(
        x
    )

    z_log_var = layers.Dense(
        latent_dim
    )(
        x
    )

    z = Sampling()(
        [
            z_mean,
            z_log_var,
        ]
    )

    encoder = models.Model(
        encoder_inputs,
        [
            z_mean,
            z_log_var,
            z,
        ],
        name="encoder",
    )

    return encoder


# ============================================================
# Decoder
# ============================================================

def build_vae_decoder(
    latent_dim=VAE_LATENT_DIM,
):

    latent_inputs = (
        layers.Input(
            shape=(
                latent_dim,
            )
        )
    )

    x = layers.Dense(
        64,
        activation="relu",
    )(
        latent_inputs
    )

    x = layers.Dense(
        256,
        activation="relu",
    )(
        x
    )

    decoder_outputs = (
        layers.Dense(
            FEATURE_DIM,
            activation="linear",
        )(
            x
        )
    )

    decoder = models.Model(
        latent_inputs,
        decoder_outputs,
        name="decoder",
    )

    return decoder


# ============================================================
# VAE model
# ============================================================

class DTI_VAE(
    keras.Model
):

    def __init__(
        self,
        encoder,
        decoder,
        **kwargs,
    ):

        super().__init__(
            **kwargs
        )

        self.encoder = (
            encoder
        )

        self.decoder = (
            decoder
        )


    def train_step(
        self,
        data,
    ):

        x = (
            data[0]
            if isinstance(
                data,
                tuple,
            )
            else data
        )

        x = tf.cast(
            x,
            tf.float32,
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with tf.GradientTape() as tape:

            (
                z_mean,
                z_log_var,
                z,
            ) = (
                self.encoder(
                    x,
                    training=True,
                )
            )

            # Immediate numerical checks
            tf.debugging.check_numerics(
                z_mean,
                "VAE z_mean contains NaN/Inf"
            )

            tf.debugging.check_numerics(
                z_log_var,
                "VAE z_log_var contains NaN/Inf"
            )

            tf.debugging.check_numerics(
                z,
                "VAE sampled latent z contains NaN/Inf"
            )

            reconstruction = (
                self.decoder(
                    z,
                    training=True,
                )
            )

            tf.debugging.check_numerics(
                reconstruction,
                "VAE reconstruction contains NaN/Inf"
            )

            # -----------------------------------------------
            # Reconstruction loss
            #
            # Continuous 100-dimensional latent features.
            # -----------------------------------------------

            reconstruction_loss = (
                tf.reduce_mean(
                    tf.reduce_sum(
                        tf.square(
                            x
                            - reconstruction
                        ),
                        axis=1,
                    )
                )
            )

            # -----------------------------------------------
            # KL divergence
            # -----------------------------------------------

            kl_loss = (
                -0.5
                * tf.reduce_mean(
                    1
                    + z_log_var
                    - tf.square(
                        z_mean
                    )
                    - tf.exp(
                        z_log_var
                    )
                )
            )

            total_loss = (
                reconstruction_loss
                + kl_loss
            )

            tf.debugging.check_numerics(
                reconstruction_loss,
                "VAE reconstruction loss contains NaN/Inf"
            )

            tf.debugging.check_numerics(
                kl_loss,
                "VAE KL loss contains NaN/Inf"
            )

            tf.debugging.check_numerics(
                total_loss,
                "VAE total loss contains NaN/Inf"
            )

        # ----------------------------------------------------
        # Gradients
        # ----------------------------------------------------

        grads = tape.gradient(
            total_loss,
            self.trainable_weights,
        )

        gradient_pairs = []

        for (
            gradient,
            variable,
        ) in zip(
            grads,
            self.trainable_weights,
        ):

            if gradient is None:

                raise RuntimeError(
                    "VAE returned a "
                    "None gradient."
                )

            tf.debugging.check_numerics(
                gradient,
                "VAE gradient contains NaN/Inf"
            )

            gradient_pairs.append(
                (
                    gradient,
                    variable,
                )
            )

        self.optimizer.apply_gradients(
            gradient_pairs
        )

        return {

            "loss": (
                total_loss
            ),

            "reconstruction_loss": (
                reconstruction_loss
            ),

            "kl_loss": (
                kl_loss
            ),
        }


# ============================================================
# Generate VAE synthetic pool
# ============================================================

def pool_vae(
    X,
    y,
):

    # --------------------------------------------------------
    # Positive INNER-TRAINING observations only
    # --------------------------------------------------------

    positive = (
        X[
            y == 1
        ]
        .astype(
            np.float32
        )
    )

    n_syn = (
        target_positive_count_for_max_pool(
            y
        )
    )

    if n_syn == 0:

        return (
            np.empty(
                (
                    0,
                    FEATURE_DIM,
                ),
                dtype=np.float32,
            ),
            {},
        )

    if len(
        positive
    ) == 0:

        raise ValueError(
            "VAE received no "
            "positive inner-training "
            "observations."
        )

    if not np.isfinite(
        positive
    ).all():

        raise FloatingPointError(
            "VAE input contains "
            "NaN or Inf before scaling."
        )

    # --------------------------------------------------------
    # Standardize positive inner-training observations
    #
    # IMPORTANT:
    # The scaler sees only role == train and y == 1.
    # --------------------------------------------------------

    scaler = (
        StandardScaler()
    )

    positive_scaled = (
        scaler
        .fit_transform(
            positive
        )
        .astype(
            np.float32
        )
    )

    if not np.isfinite(
        positive_scaled
    ).all():

        raise FloatingPointError(
            "VAE standardized "
            "training features contain "
            "NaN or Inf."
        )

    print(
        "VAE positive input range "
        f"before scaling: "
        f"[{positive.min():.6f}, "
        f"{positive.max():.6f}]"
    )

    print(
        "VAE positive input range "
        f"after scaling: "
        f"[{positive_scaled.min():.6f}, "
        f"{positive_scaled.max():.6f}]"
    )

    # --------------------------------------------------------
    # New TensorFlow state for this VAE fit
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    encoder = (
        build_vae_encoder()
    )

    decoder = (
        build_vae_decoder()
    )

    vae = DTI_VAE(
        encoder,
        decoder,
    )

    vae.compile(
        optimizer=(
            tf.keras.optimizers.Adam()
        )
    )

    effective_batch_size = min(
        VAE_BATCH_SIZE,
        len(
            positive_scaled
        ),
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = vae.fit(
        positive_scaled,
        epochs=(
            VAE_EPOCHS
        ),
        batch_size=(
            effective_batch_size
        ),
        verbose=1,
    )

    # --------------------------------------------------------
    # Validate training history
    # --------------------------------------------------------

    final_loss = None
    final_reconstruction_loss = None
    final_kl_loss = None

    if (
        "loss"
        in history.history
    ):

        final_loss = float(
            history.history[
                "loss"
            ][-1]
        )

        if not np.isfinite(
            final_loss
        ):

            raise FloatingPointError(
                "VAE final loss "
                "is NaN or Inf."
            )

    if (
        "reconstruction_loss"
        in history.history
    ):

        final_reconstruction_loss = (
            float(
                history.history[
                    "reconstruction_loss"
                ][-1]
            )
        )

        if not np.isfinite(
            final_reconstruction_loss
        ):

            raise FloatingPointError(
                "VAE final "
                "reconstruction loss "
                "is NaN or Inf."
            )

    if (
        "kl_loss"
        in history.history
    ):

        final_kl_loss = float(
            history.history[
                "kl_loss"
            ][-1]
        )

        if not np.isfinite(
            final_kl_loss
        ):

            raise FloatingPointError(
                "VAE final KL loss "
                "is NaN or Inf."
            )

    # --------------------------------------------------------
    # Generate latent samples
    #
    # Source implementation did not fix the NumPy seed.
    # --------------------------------------------------------

    z_new = (
        np.random.normal(
            size=(
                n_syn,
                VAE_LATENT_DIM,
            )
        )
        .astype(
            np.float32
        )
    )

    # --------------------------------------------------------
    # Generate in batches
    # --------------------------------------------------------

    synthetic_scaled = (
        decoder.predict(
            z_new,
            batch_size=1024,
            verbose=0,
        )
        .astype(
            np.float32
        )
    )

    if not np.isfinite(
        synthetic_scaled
    ).all():

        raise FloatingPointError(
            "VAE generated "
            "NaN/Inf values in "
            "standardized feature space."
        )

    # --------------------------------------------------------
    # Return generated observations to original
    # latent-feature scale
    # --------------------------------------------------------

    synthetic = (
        scaler
        .inverse_transform(
            synthetic_scaled
        )
        .astype(
            np.float32
        )
    )

    if not np.isfinite(
        synthetic
    ).all():

        raise FloatingPointError(
            "VAE generated "
            "NaN/Inf values after "
            "inverse scaling."
        )

    # --------------------------------------------------------
    # Shape audit
    # --------------------------------------------------------

    expected_shape = (
        n_syn,
        FEATURE_DIM,
    )

    if (
        synthetic.shape
        != expected_shape
    ):

        raise AssertionError(
            f"Unexpected VAE "
            f"synthetic shape: "
            f"{synthetic.shape}; "
            f"expected "
            f"{expected_shape}."
        )

    # --------------------------------------------------------
    # Training information
    # --------------------------------------------------------

    training_info = {

        "epochs": (
            VAE_EPOCHS
        ),

        "nominal_batch_size": (
            VAE_BATCH_SIZE
        ),

        "effective_batch_size": (
            effective_batch_size
        ),

        "latent_dimension": (
            VAE_LATENT_DIM
        ),

        "optimizer": (
            "Adam"
        ),

        "reconstruction_loss": (
            "squared error"
        ),

        "standardized_training_features": (
            True
        ),

        "scaler_fit_scope": (
            "positive inner-training "
            "observations only"
        ),

        "final_loss": (
            final_loss
        ),

        "final_reconstruction_loss": (
            final_reconstruction_loss
        ),

        "final_kl_loss": (
            final_kl_loss
        ),
    }

    return (
        synthetic,
        training_info,
    )

## 6. WGAN-GP augmentation



In [7]:
# ============================================================
# WGAN-GP AUGMENTATION
# ============================================================

def make_generator_model():

    return tf.keras.Sequential([
        layers.Input(
            shape=(WGAN_NOISE_DIM,)
        ),

        layers.Dense(
            256,
            activation="relu",
        ),

        layers.Dense(
            512,
            activation="relu",
        ),

        layers.Dense(
            1024,
            activation="relu",
        ),

        layers.Dense(
            FEATURE_DIM,
            activation="linear",
        ),
    ])


def make_critic_model():

    return tf.keras.Sequential([
        layers.Input(
            shape=(FEATURE_DIM,)
        ),

        layers.Dense(
            512,
            activation="relu",
        ),

        layers.Dropout(
            0.1
        ),

        layers.Dense(
            256,
            activation="relu",
        ),

        layers.Dense(
            1
        ),
    ])


def critic_loss(
    real_output,
    fake_output,
):

    return (
        tf.reduce_mean(
            fake_output
        )
        - tf.reduce_mean(
            real_output
        )
    )


def generator_loss(
    fake_output,
):

    return -tf.reduce_mean(
        fake_output
    )


def gradient_penalty(
    real_data,
    fake_data,
    critic,
):

    batch_size = tf.shape(
        real_data
    )[0]

    # Interpolation coefficient must remain in [0, 1].
    epsilon = tf.random.uniform(
        shape=[
            batch_size,
            1,
        ],
        minval=0.0,
        maxval=1.0,
    )

    interpolated = (
        epsilon
        * real_data
        + (
            1.0
            - epsilon
        )
        * fake_data
    )

    with tf.GradientTape() as tape:

        tape.watch(
            interpolated
        )

        prediction = critic(
            interpolated,
            training=True,
        )

    gradients = tape.gradient(
        prediction,
        interpolated,
    )

    gradient_norm = tf.sqrt(
        tf.reduce_sum(
            tf.square(
                gradients
            ),
            axis=1,
        )
        + 1e-12
    )

    penalty = tf.reduce_mean(
        (
            gradient_norm
            - 1.0
        )
        ** 2
    )

    return penalty


def pool_wgangp(
    X,
    y,
):

    # ========================================================
    # Positive inner-training observations only
    # ========================================================

    positive = (
        X[
            y == 1
        ]
        .astype(
            np.float32
        )
    )

    n_syn = (
        target_positive_count_for_max_pool(
            y
        )
    )

    if n_syn == 0:

        return (
            np.empty(
                (
                    0,
                    FEATURE_DIM,
                ),
                dtype=np.float32,
            ),
            {},
        )

    if len(
        positive
    ) == 0:

        raise ValueError(
            "WGAN-GP received no "
            "positive training observations."
        )

    # ========================================================
    # Numerical stabilization
    #
    # Scaler is fitted ONLY on positive inner-training data.
    # ========================================================

    scaler = StandardScaler()

    positive_scaled = (
        scaler
        .fit_transform(
            positive
        )
        .astype(
            np.float32
        )
    )

    if not np.isfinite(
        positive_scaled
    ).all():

        raise FloatingPointError(
            "Non-finite values detected "
            "after WGAN-GP training-data scaling."
        )

    # ========================================================
    # New TensorFlow state for this fit
    # ========================================================

    tf.keras.backend.clear_session()

    generator = (
        make_generator_model()
    )

    critic = (
        make_critic_model()
    )

    generator_optimizer = Adam(
        learning_rate=(
            WGAN_LEARNING_RATE
        ),
        beta_1=(
            WGAN_BETA_1
        ),
    )

    critic_optimizer = Adam(
        learning_rate=(
            WGAN_LEARNING_RATE
        ),
        beta_1=(
            WGAN_BETA_1
        ),
    )

    # Explicitly build optimizer slot variables before tf.function.
    # This avoids:
    # "tf.function only supports singleton tf.Variables..."
    generator_optimizer.build(
        generator.trainable_variables
    )

    critic_optimizer.build(
        critic.trainable_variables
    )

    # ========================================================
    # Training step belongs to THIS generator/critic pair
    # ========================================================

    @tf.function
    def train_step(
        real_features,
    ):

        batch_size = tf.shape(
            real_features
        )[0]

        noise = tf.random.normal(
            shape=[
                batch_size,
                WGAN_NOISE_DIM,
            ]
        )

        with (
            tf.GradientTape()
            as generator_tape,
            tf.GradientTape()
            as critic_tape,
        ):

            generated = generator(
                noise,
                training=True,
            )

            real_output = critic(
                real_features,
                training=True,
            )

            fake_output = critic(
                generated,
                training=True,
            )

            base_critic_loss = (
                critic_loss(
                    real_output,
                    fake_output,
                )
            )

            gp = gradient_penalty(
                real_features,
                generated,
                critic,
            )

            total_critic_loss = (
                base_critic_loss
                + (
                    WGAN_GP_LAMBDA
                    * gp
                )
            )

            gen_loss = (
                generator_loss(
                    fake_output
                )
            )

            # Fail before applying gradients if losses diverge.
            tf.debugging.check_numerics(
                total_critic_loss,
                "WGAN critic loss contains NaN/Inf"
            )

            tf.debugging.check_numerics(
                gen_loss,
                "WGAN generator loss contains NaN/Inf"
            )

        generator_gradients = (
            generator_tape.gradient(
                gen_loss,
                generator.trainable_variables,
            )
        )

        critic_gradients = (
            critic_tape.gradient(
                total_critic_loss,
                critic.trainable_variables,
            )
        )

        generator_pairs = []

        for gradient, variable in zip(
            generator_gradients,
            generator.trainable_variables,
        ):

            if gradient is None:

                raise RuntimeError(
                    "WGAN generator returned "
                    "a None gradient."
                )

            tf.debugging.check_numerics(
                gradient,
                "WGAN generator gradient "
                "contains NaN/Inf"
            )

            generator_pairs.append(
                (
                    gradient,
                    variable,
                )
            )

        critic_pairs = []

        for gradient, variable in zip(
            critic_gradients,
            critic.trainable_variables,
        ):

            if gradient is None:

                raise RuntimeError(
                    "WGAN critic returned "
                    "a None gradient."
                )

            tf.debugging.check_numerics(
                gradient,
                "WGAN critic gradient "
                "contains NaN/Inf"
            )

            critic_pairs.append(
                (
                    gradient,
                    variable,
                )
            )

        generator_optimizer.apply_gradients(
            generator_pairs
        )

        critic_optimizer.apply_gradients(
            critic_pairs
        )

        return (
            total_critic_loss,
            gen_loss,
        )

    # ========================================================
    # Dataset
    # ========================================================

    effective_batch_size = min(
        WGAN_BATCH_SIZE,
        len(
            positive_scaled
        ),
    )

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            positive_scaled
        )
        .shuffle(
            buffer_size=len(
                positive_scaled
            ),
            reshuffle_each_iteration=True,
        )
        .batch(
            effective_batch_size,
            drop_remainder=False,
        )
    )

    final_critic_loss = None
    final_generator_loss = None

    # ========================================================
    # Train
    # ========================================================

    for epoch in range(
        WGAN_EPOCHS
    ):

        for real_batch in dataset:

            (
                final_critic_loss,
                final_generator_loss,
            ) = train_step(
                real_batch
            )

        critic_value = float(
            final_critic_loss.numpy()
        )

        generator_value = float(
            final_generator_loss.numpy()
        )

        if not (
            np.isfinite(
                critic_value
            )
            and np.isfinite(
                generator_value
            )
        ):

            raise FloatingPointError(
                "WGAN-GP numerical "
                f"divergence at epoch "
                f"{epoch + 1}: "
                f"critic_loss="
                f"{critic_value}, "
                f"generator_loss="
                f"{generator_value}"
            )

        if (
            epoch == 0
            or (
                epoch + 1
            )
            % 10
            == 0
        ):

            print(
                f"WGAN epoch "
                f"{epoch + 1:3d} | "
                f"critic="
                f"{critic_value:.6f} | "
                f"generator="
                f"{generator_value:.6f}"
            )

    # ========================================================
    # Generate maximum synthetic pool
    # ========================================================

    synthetic_batches = []

    remaining = int(
        n_syn
    )

    generation_batch_size = 4096

    while remaining > 0:

        current = min(
            generation_batch_size,
            remaining,
        )

        noise = tf.random.normal(
            shape=[
                current,
                WGAN_NOISE_DIM,
            ]
        )

        generated_scaled = (
            generator(
                noise,
                training=False,
            )
            .numpy()
        )

        if not np.isfinite(
            generated_scaled
        ).all():

            raise FloatingPointError(
                "WGAN-GP generated "
                "NaN/Inf values before "
                "inverse scaling."
            )

        generated = (
            scaler
            .inverse_transform(
                generated_scaled
            )
            .astype(
                np.float32
            )
        )

        if not np.isfinite(
            generated
        ).all():

            raise FloatingPointError(
                "WGAN-GP generated "
                "NaN/Inf values after "
                "inverse scaling."
            )

        synthetic_batches.append(
            generated
        )

        remaining -= current

    synthetic = np.vstack(
        synthetic_batches
    ).astype(
        np.float32
    )

    if synthetic.shape != (
        n_syn,
        FEATURE_DIM,
    ):

        raise AssertionError(
            "Unexpected WGAN-GP "
            f"synthetic shape: "
            f"{synthetic.shape}; "
            f"expected "
            f"({n_syn}, {FEATURE_DIM})."
        )

    if not np.isfinite(
        synthetic
    ).all():

        raise FloatingPointError(
            "Final WGAN-GP pool "
            "contains NaN/Inf."
        )

    # ========================================================
    # Training metadata
    # ========================================================

    training_info = {

        "epochs": (
            WGAN_EPOCHS
        ),

        "nominal_batch_size": (
            WGAN_BATCH_SIZE
        ),

        "effective_batch_size": (
            effective_batch_size
        ),

        "learning_rate": (
            WGAN_LEARNING_RATE
        ),

        "beta_1": (
            WGAN_BETA_1
        ),

        "gradient_penalty_lambda": (
            WGAN_GP_LAMBDA
        ),

        "standardized_training_features": (
            True
        ),

        "scaler_fit_scope": (
            "positive inner-training "
            "observations only"
        ),

        "final_critic_loss": float(
            final_critic_loss.numpy()
        ),

        "final_generator_loss": float(
            final_generator_loss.numpy()
        ),
    }

    return (
        synthetic,
        training_info,
    )

## 7. Latent Diffusion augmentation



In [8]:
class DiffusionBetaVAE(tf.keras.Model):
    def __init__(
        self,
        input_dim,
        latent_dim=DIFFUSION_VAE_LATENT_DIM,
    ):
        super().__init__()

        self.latent_dim = latent_dim

        self.encoder = models.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(
                128,
                activation="relu",
            ),
            layers.Dense(
                64,
                activation="relu",
            ),
            layers.Dense(
                latent_dim * 2,
            ),
        ])

        self.decoder = models.Sequential([
            layers.Input(
                shape=(latent_dim,)
            ),
            layers.Dense(
                64,
                activation="relu",
            ),
            layers.Dense(
                128,
                activation="relu",
            ),
            # MSCMF latent features are not bounded to [0, 1].
            layers.Dense(
                input_dim,
                activation="linear",
            ),
        ])

    def reparameterize(
        self,
        mean,
        logvar,
    ):
        eps = tf.random.normal(
            shape=tf.shape(mean)
        )
        return (
            mean
            + tf.exp(0.5 * logvar)
            * eps
        )

    def call(
        self,
        x,
        training=False,
    ):
        z_params = self.encoder(
            x,
            training=training,
        )

        mean, logvar = tf.split(
            z_params,
            num_or_size_splits=2,
            axis=1,
        )

        z = self.reparameterize(
            mean,
            logvar,
        )

        x_recon = self.decoder(
            z,
            training=training,
        )

        return (
            x_recon,
            mean,
            logvar,
            z,
        )


class ConditionalDiffusionModel(tf.keras.Model):
    def __init__(
        self,
        latent_dim,
        cond_dim,
    ):
        super().__init__()

        self.model = models.Sequential([
            layers.Input(
                shape=(
                    latent_dim
                    + cond_dim
                    + 1,
                )
            ),
            layers.Dense(
                128,
                activation="swish",
            ),
            layers.Dense(
                128,
                activation="swish",
            ),
            layers.Dense(
                latent_dim,
            ),
        ])

    def call(
        self,
        x,
        cond,
        t,
        training=False,
    ):
        t_embed = (
            tf.cast(
                t,
                tf.float32,
            )
            / float(
                DIFFUSION_TIMESTEPS
            )
        )

        t_embed = tf.expand_dims(
            t_embed,
            axis=-1,
        )

        x_cat = tf.concat(
            [
                x,
                cond,
                t_embed,
            ],
            axis=-1,
        )

        return self.model(
            x_cat,
            training=training,
        )


def diffusion_schedule():
    betas = np.linspace(
        DIFFUSION_BETA_START,
        DIFFUSION_BETA_END,
        DIFFUSION_TIMESTEPS,
        dtype=np.float32,
    )

    alphas = 1.0 - betas
    alphas_cumprod = np.cumprod(
        alphas
    )

    return {
        "betas": tf.constant(
            betas,
            dtype=tf.float32,
        ),
        "alphas": tf.constant(
            alphas,
            dtype=tf.float32,
        ),
        "alphas_cumprod": tf.constant(
            alphas_cumprod,
            dtype=tf.float32,
        ),
        "sqrt_alphas_cumprod": (
            tf.constant(
                np.sqrt(
                    alphas_cumprod
                ),
                dtype=tf.float32,
            )
        ),
        "sqrt_one_minus_alphas_cumprod": (
            tf.constant(
                np.sqrt(
                    1.0
                    - alphas_cumprod
                ),
                dtype=tf.float32,
            )
        ),
    }


def diffusion_q_sample(
    x_start,
    t,
    noise,
    schedule,
):
    sqrt_alpha_bar = tf.gather(
        schedule[
            "sqrt_alphas_cumprod"
        ],
        t,
    )[:, None]

    sqrt_one_minus = tf.gather(
        schedule[
            "sqrt_one_minus_alphas_cumprod"
        ],
        t,
    )[:, None]

    return (
        sqrt_alpha_bar
        * x_start
        + sqrt_one_minus
        * noise
    )


def train_diffusion_vae(
    positive,
):
    vae = DiffusionBetaVAE(
        input_dim=FEATURE_DIM,
        latent_dim=(
            DIFFUSION_VAE_LATENT_DIM
        ),
    )

    optimizer = (
        tf.keras.optimizers.Adam(
            learning_rate=(
                DIFFUSION_VAE_LEARNING_RATE
            )
        )
    )

    x = tf.convert_to_tensor(
        positive,
        dtype=tf.float32,
    )

    final_loss = None

    for _ in range(
        DIFFUSION_VAE_EPOCHS
    ):
        with tf.GradientTape() as tape:
            (
                x_recon,
                mean,
                logvar,
                z,
            ) = vae(
                x,
                training=True,
            )

            recon_loss = tf.reduce_mean(
                tf.square(
                    x - x_recon
                )
            )

            kl_loss = (
                -0.5
                * tf.reduce_mean(
                    1
                    + logvar
                    - tf.square(mean)
                    - tf.exp(logvar)
                )
            )

            final_loss = (
                recon_loss
                + DIFFUSION_VAE_BETA
                * kl_loss
            )

        grads = tape.gradient(
            final_loss,
            vae.trainable_variables,
        )

        optimizer.apply_gradients(
            zip(
                grads,
                vae.trainable_variables,
            )
        )

    (
        _,
        mean,
        logvar,
        latent,
    ) = vae(
        x,
        training=False,
    )

    return (
        vae,
        latent,
        float(
            final_loss.numpy()
        ),
    )


def train_conditional_diffusion(
    latent,
    positive,
):
    schedule = (
        diffusion_schedule()
    )

    model = ConditionalDiffusionModel(
        latent_dim=(
            DIFFUSION_VAE_LATENT_DIM
        ),
        cond_dim=FEATURE_DIM,
    )

    optimizer = (
        tf.keras.optimizers.Adam(
            DIFFUSION_LEARNING_RATE
        )
    )

    cond_tensor = (
        tf.convert_to_tensor(
            positive,
            dtype=tf.float32,
        )
    )

    latent_tensor = (
        tf.convert_to_tensor(
            latent,
            dtype=tf.float32,
        )
    )

    batch_size = tf.shape(
        latent_tensor
    )[0]

    final_loss = None

    for _ in range(
        DIFFUSION_EPOCHS
    ):
        t = tf.random.uniform(
            (batch_size,),
            minval=0,
            maxval=(
                DIFFUSION_TIMESTEPS
            ),
            dtype=tf.int32,
        )

        noise = tf.random.normal(
            shape=tf.shape(
                latent_tensor
            )
        )

        with tf.GradientTape() as tape:
            x_noisy = (
                diffusion_q_sample(
                    latent_tensor,
                    t,
                    noise,
                    schedule,
                )
            )

            noise_pred = model(
                x_noisy,
                cond_tensor,
                t,
                training=True,
            )

            final_loss = (
                tf.reduce_mean(
                    tf.square(
                        noise
                        - noise_pred
                    )
                )
            )

        grads = tape.gradient(
            final_loss,
            model.trainable_variables,
        )

        optimizer.apply_gradients(
            zip(
                grads,
                model.trainable_variables,
            )
        )

    return (
        model,
        schedule,
        float(
            final_loss.numpy()
        ),
    )


def reverse_diffusion_sample(
    model,
    cond,
    n_samples,
    schedule,
):
    x = tf.random.normal(
        (
            n_samples,
            DIFFUSION_VAE_LATENT_DIM,
        )
    )

    # Retain the source notebook's 100-step
    # reverse-generation horizon.
    for t_val in reversed(
        range(
            DIFFUSION_SAMPLING_STEPS
        )
    ):
        t = tf.fill(
            (n_samples,),
            t_val,
        )

        predicted_noise = model(
            x,
            cond,
            t,
            training=False,
        )

        alpha_t = tf.gather(
            schedule["alphas"],
            t,
        )[:, None]

        alpha_bar_t = tf.gather(
            schedule[
                "alphas_cumprod"
            ],
            t,
        )[:, None]

        beta_t = tf.gather(
            schedule["betas"],
            t,
        )[:, None]

        mean = (
            1.0
            / tf.sqrt(alpha_t)
            * (
                x
                - (
                    beta_t
                    / tf.sqrt(
                        1.0
                        - alpha_bar_t
                    )
                )
                * predicted_noise
            )
        )

        if t_val > 0:
            noise = tf.random.normal(
                shape=tf.shape(x)
            )
            x = (
                mean
                + tf.sqrt(beta_t)
                * noise
            )
        else:
            x = mean

    return x


def pool_latent_diffusion(
    X,
    y,
):
    positive = X[
        y == 1
    ].astype(np.float32)

    n_syn = (
        target_positive_count_for_max_pool(
            y
        )
    )

    if n_syn == 0:
        return (
            np.empty(
                (0, FEATURE_DIM),
                dtype=np.float32,
            ),
            {},
        )

    tf.keras.backend.clear_session()

    (
        vae,
        latent,
        final_vae_loss,
    ) = train_diffusion_vae(
        positive
    )

    (
        diffusion_model,
        schedule,
        final_diffusion_loss,
    ) = (
        train_conditional_diffusion(
            latent,
            positive,
        )
    )

    synthetic_batches = []
    remaining = n_syn

    # Batched generation avoids creating the entire
    # conditioning/generation tensor at once.
    generation_batch_size = 2048

    while remaining > 0:
        current = min(
            generation_batch_size,
            remaining,
        )

        # Conditions are drawn only from positive
        # inner-training observations.
        cond_indices = np.random.choice(
            len(positive),
            size=current,
            replace=True,
        )

        cond = tf.convert_to_tensor(
            positive[
                cond_indices
            ],
            dtype=tf.float32,
        )

        synthetic_latent = (
            reverse_diffusion_sample(
                diffusion_model,
                cond,
                current,
                schedule,
            )
        )

        decoded = vae.decoder(
            synthetic_latent,
            training=False,
        ).numpy()

        synthetic_batches.append(
            decoded.astype(
                np.float32
            )
        )

        remaining -= current

    synthetic = np.vstack(
        synthetic_batches
    )

    training_info = {
        "vae_latent_dim": (
            DIFFUSION_VAE_LATENT_DIM
        ),
        "vae_epochs": (
            DIFFUSION_VAE_EPOCHS
        ),
        "vae_learning_rate": (
            DIFFUSION_VAE_LEARNING_RATE
        ),
        "vae_beta": (
            DIFFUSION_VAE_BETA
        ),
        "diffusion_epochs": (
            DIFFUSION_EPOCHS
        ),
        "timesteps": (
            DIFFUSION_TIMESTEPS
        ),
        "sampling_steps": (
            DIFFUSION_SAMPLING_STEPS
        ),
        "diffusion_learning_rate": (
            DIFFUSION_LEARNING_RATE
        ),
        "beta_start": (
            DIFFUSION_BETA_START
        ),
        "beta_end": (
            DIFFUSION_BETA_END
        ),
        "final_vae_loss": (
            final_vae_loss
        ),
        "final_diffusion_loss": (
            final_diffusion_loss
        ),
    }

    return (
        synthetic,
        training_info,
    )

## 8. Method dispatcher and compact save/load utilities

In [9]:
# ============================================================
# METHOD METADATA
# ============================================================

def method_metadata(
    backbone,
    method,
):

    common = {

        "implementation_version": (
            IMPLEMENTATION_VERSION
        ),

        "backbone": (
            backbone
        ),

        "method": (
            method
        ),

        "feature_dimension": (
            FEATURE_DIM
        ),

        "train_only": (
            True
        ),
    }

    details = {

        # ====================================================
        # Random Over-Sampling
        # ====================================================

        "RandomOver": {

            "random_state": (
                RANDOM_STATE
            ),

            "replace": (
                True
            ),
        },

        # ====================================================
        # Random Under-Sampling
        # ====================================================

        "RandomUnder": {

            "random_state": (
                RANDOM_STATE
            ),

            "replace": (
                False
            ),
        },

        # ====================================================
        # Tomek Links
        # ====================================================

        "TomekLinks": {},

        # ====================================================
        # CNN + Tomek Links
        # ====================================================

        "CNNTomekLinks": {

            "standard_scaler": (
                True
            ),

            "cnn_n_neighbors": (
                1
            ),

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # SMOTE
        # ====================================================

        "SMOTE": {

            "standard_scaler": (
                True
            ),

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # SMOTE + Tomek Links
        # ====================================================

        "SMOTETomekLinks": {

            "standard_scaler": (
                True
            ),

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # Basic SMOTE
        # ====================================================

        "BasicSMOTE": {

            "standard_scaler": (
                False
            ),

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # ADASYN
        # ====================================================

        "ADASYN": {

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # KMeans-SMOTE
        # ====================================================

        "KMeansSMOTE": {

            "n_clusters": (
                5
            ),

            "cluster_balance_threshold": (
                0.01
            ),

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # Latent Sampling / GMM
        # ====================================================

        "LatentSampling": {

            "model": (
                "GaussianMixture"
            ),

            "n_components": (
                1
            ),

            "covariance_type": (
                "full"
            ),

            "random_state": (
                RANDOM_STATE
            ),
        },

        # ====================================================
        # Gaussian Noise Injection
        # ====================================================

        "NoiseInjection": {

            "sigma": (
                NOISE_SIGMA
            ),

            "fixed_numpy_seed": (
                False
            ),
        },

        # ====================================================
        # VAE
        # ====================================================

        "VAE": {

            "latent_dim": (
                VAE_LATENT_DIM
            ),

            "epochs": (
                VAE_EPOCHS
            ),

            "batch_size": (
                VAE_BATCH_SIZE
            ),

            "optimizer": (
                "Adam"
            ),

            "reconstruction_loss": (
                "squared error"
            ),

            # New numerical-stability preprocessing.
            "standard_scaler": (
                True
            ),

            "scaler_fit_scope": (
                "positive inner-training "
                "observations only"
            ),
        },

        # ====================================================
        # WGAN-GP
        # ====================================================

        "WGANGP": {

            "noise_dim": (
                WGAN_NOISE_DIM
            ),

            "epochs": (
                WGAN_EPOCHS
            ),

            "batch_size": (
                WGAN_BATCH_SIZE
            ),

            "learning_rate": (
                WGAN_LEARNING_RATE
            ),

            "beta_1": (
                WGAN_BETA_1
            ),

            "gradient_penalty_lambda": (
                WGAN_GP_LAMBDA
            ),

            "standard_scaler": (
                True
            ),

            "scaler_fit_scope": (
                "positive inner-training "
                "observations only"
            ),
        },

        # ====================================================
        # Latent Diffusion
        # ====================================================

        "LatentDiffusion": {

            "input_backbone": (
                "MSCMF"
            ),

            "vae_latent_dim": (
                DIFFUSION_VAE_LATENT_DIM
            ),

            "vae_epochs": (
                DIFFUSION_VAE_EPOCHS
            ),

            "vae_learning_rate": (
                DIFFUSION_VAE_LEARNING_RATE
            ),

            "vae_beta": (
                DIFFUSION_VAE_BETA
            ),

            "diffusion_epochs": (
                DIFFUSION_EPOCHS
            ),

            "timesteps": (
                DIFFUSION_TIMESTEPS
            ),

            "sampling_steps": (
                DIFFUSION_SAMPLING_STEPS
            ),

            "diffusion_learning_rate": (
                DIFFUSION_LEARNING_RATE
            ),

            "beta_start": (
                DIFFUSION_BETA_START
            ),

            "beta_end": (
                DIFFUSION_BETA_END
            ),
        },
    }

    if (
        method
        not in details
    ):

        raise ValueError(
            f"Unknown augmentation "
            f"method: {method}"
        )

    common[
        "parameters"
    ] = (
        details[
            method
        ]
    )

    return common

## 9. Generate all candidate pools and exact-ratio plans


Completed compatible pools are reused when `RESUME_IF_VALID=True`.

In [10]:
INPUT_AUDIT_ROWS = []
POOL_STATS_ROWS = []
RATIO_AUDIT_ROWS = []

for family, cfg in FAMILIES.items():

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        # Cache each backbone once per fold.
        backbone_data = {}

        for backbone in BACKBONES:
            backbone_data[backbone] = (
                load_training_representation(
                    family,
                    outer_fold,
                    backbone,
                )
            )

        # Cross-backbone role integrity.
        reference_ids = (
            backbone_data["MSCMF"][
                "train_roles"
            ]["pair_id"].to_numpy()
        )

        for backbone in BACKBONES:
            assert np.array_equal(
                reference_ids,
                backbone_data[backbone][
                    "train_roles"
                ]["pair_id"].to_numpy(),
            )

        source_ref = backbone_data[
            "MSCMF"
        ]

        roles_all = source_ref[
            "roles_all"
        ]
        train_roles = source_ref[
            "train_roles"
        ]

        train_ids = set(
            train_roles["pair_id"]
        )
        val_ids = set(
            roles_all.loc[
                roles_all["role"].eq(
                    "validation"
                ),
                "pair_id",
            ]
        )
        test_ids = set(
            roles_all.loc[
                roles_all["role"].eq(
                    "test"
                ),
                "pair_id",
            ]
        )

        assert not (
            train_ids & val_ids
        )
        assert not (
            train_ids & test_ids
        )

        INPUT_AUDIT_ROWS.append({
            "family": cfg["display"],
            "outer_fold": outer_fold,
            "n_train": len(train_roles),
            "n_validation": len(val_ids),
            "n_test": len(test_ids),
            "train_validation_overlap": 0,
            "train_test_overlap": 0,
            "all_backbones_same_train_pair_ids": True,
            "pass": True,
        })

        for backbone, method in (
            METHOD_BACKBONE_CONFIGS
        ):

            source = backbone_data[
                backbone
            ]
            X = source["X"]
            y = source["y"]

            out_dir = augmentation_dir(
                family,
                outer_fold,
                backbone,
                method,
            )
            out_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            print(
                "\n"
                + "=" * 88
            )
            print(
                f"{cfg['display']} | "
                f"fold {outer_fold} | "
                f"{backbone} | "
                f"{method}"
            )
            print(
                "=" * 88
            )

            resumed = False

            if (
                RESUME_IF_VALID
                and valid_saved_pool(
                    family,
                    outer_fold,
                    backbone,
                    method,
                    source,
                )
            ):
                (
                    real_allowed,
                    synthetic,
                    metadata,
                ) = load_candidate_pool(
                    family,
                    outer_fold,
                    backbone,
                    method,
                )
                elapsed = metadata.get(
                    "elapsed_seconds",
                    0.0,
                )
                resumed = True

            else:
                start = time.time()

                (
                    real_allowed,
                    synthetic,
                    training_info,
                ) = create_candidate_pool(
                    X,
                    y,
                    method,
                )

                elapsed = (
                    time.time() - start
                )

                save_candidate_pool(
                    family,
                    outer_fold,
                    backbone,
                    method,
                    source,
                    real_allowed,
                    synthetic,
                    training_info,
                    elapsed,
                )

            real_allowed = np.asarray(
                real_allowed,
                dtype=np.int64,
            )
            synthetic = np.asarray(
                synthetic,
                dtype=np.float32,
            )

            assert (
                real_allowed.ndim == 1
            )
            assert (
                (real_allowed >= 0).all()
            )
            assert (
                (
                    real_allowed
                    < len(X)
                ).all()
            )
            assert (
                len(np.unique(real_allowed))
                == len(real_allowed)
            )

            assert (
                synthetic.ndim == 2
            )
            assert (
                synthetic.shape[1]
                == FEATURE_DIM
            )
            if not np.isfinite(synthetic).all():
                print("\nNON-FINITE SYNTHETIC DATA")
                print("Family   :", cfg["display"])
                print("Fold     :", outer_fold)
                print("Backbone :", backbone)
                print("Method   :", method)
                print("Shape    :", synthetic.shape)
                print("NaN      :", np.isnan(synthetic).sum())
                print("+Inf     :", np.isposinf(synthetic).sum())
                print("-Inf     :", np.isneginf(synthetic).sum())


            assert np.isfinite(
                synthetic
            ).all()

            candidate_y_real = y[
                real_allowed
            ]

            POOL_STATS_ROWS.append({
                "family": cfg["display"],
                "outer_fold": outer_fold,
                "backbone": backbone,
                "method": method,
                "n_train_input": len(y),
                "n_train_positive": int(
                    (y == 1).sum()
                ),
                "n_train_non_interaction": int(
                    (y == 0).sum()
                ),
                "n_real_allowed": int(
                    len(real_allowed)
                ),
                "n_real_allowed_positive": int(
                    (
                        candidate_y_real
                        == 1
                    ).sum()
                ),
                "n_real_allowed_non_interaction": int(
                    (
                        candidate_y_real
                        == 0
                    ).sum()
                ),
                "n_synthetic_positive_pool": int(
                    len(synthetic)
                ),
                "feature_dimension": int(
                    synthetic.shape[1]
                ),
                "finite_synthetic_pass": bool(
                    np.isfinite(
                        synthetic
                    ).all()
                ),
                "resumed_existing_pool": (
                    resumed
                ),
                "elapsed_seconds": elapsed,
            })

            for ratio_name in TARGET_RATIOS:

                plan = exact_ratio_plan(
                    y_real=y,
                    real_allowed_indices=(
                        real_allowed
                    ),
                    n_synthetic_positive=(
                        len(synthetic)
                    ),
                    ratio_name=ratio_name,
                )

                ratio_file = (
                    out_dir
                    / f"ratio_{ratio_name}_plan.npz"
                )

                np.savez_compressed(
                    ratio_file,
                    real_indices=plan[
                        "real_indices"
                    ],
                    synthetic_indices=plan[
                        "synthetic_indices"
                    ],
                )

                neg_units, pos_units = (
                    TARGET_RATIOS[
                        ratio_name
                    ]
                )

                exact_ratio_pass = (
                    plan["n0"]
                    * pos_units
                    == plan["n1"]
                    * neg_units
                )

                selected_train_pair_ids = (
                    train_roles.iloc[
                        plan["real_indices"]
                    ]["pair_id"]
                )

                leakage_pass = (
                    set(
                        selected_train_pair_ids
                    ).isdisjoint(
                        val_ids
                    )
                    and set(
                        selected_train_pair_ids
                    ).isdisjoint(
                        test_ids
                    )
                )

                assert exact_ratio_pass
                assert leakage_pass

                RATIO_AUDIT_ROWS.append({
                    "family": cfg["display"],
                    "outer_fold": outer_fold,
                    "backbone": backbone,
                    "method": method,
                    "ratio": ratio_name,
                    "target_non_interaction_units": neg_units,
                    "target_interaction_units": pos_units,
                    "n_non_interaction": plan[
                        "n0"
                    ],
                    "n_interaction": plan[
                        "n1"
                    ],
                    "n_total": plan[
                        "n_total"
                    ],
                    "n_real": plan[
                        "n_real"
                    ],
                    "n_synthetic": plan[
                        "n_synthetic"
                    ],
                    "exact_ratio_pass": (
                        exact_ratio_pass
                    ),
                    "validation_pair_ids_used": 0,
                    "test_pair_ids_used": 0,
                    "train_only_pass": (
                        leakage_pass
                    ),
                    "feature_dimension": FEATURE_DIM,
                    "pass": (
                        exact_ratio_pass
                        and leakage_pass
                    ),
                })

            # Save progress after each configuration.
            pd.DataFrame(
                POOL_STATS_ROWS
            ).to_excel(
                STATS_DIR
                / "augmentation_pool_statistics.xlsx",
                index=False,
                sheet_name="Pool Statistics",
            )

            pd.DataFrame(
                RATIO_AUDIT_ROWS
            ).to_excel(
                STATS_DIR
                / "augmentation_ratio_audit.xlsx",
                index=False,
                sheet_name="Ratio Audit",
            )

            print(
                f"PASS - {cfg['display']} | "
                f"fold {outer_fold} | "
                f"{backbone} | {method}"
            )

INPUT_AUDIT = pd.DataFrame(
    INPUT_AUDIT_ROWS
)
POOL_STATS = pd.DataFrame(
    POOL_STATS_ROWS
)
RATIO_AUDIT = pd.DataFrame(
    RATIO_AUDIT_ROWS
)

INPUT_AUDIT.to_excel(
    STATS_DIR
    / "augmentation_input_audit.xlsx",
    index=False,
    sheet_name="Input Audit",
)

display(INPUT_AUDIT)
display(POOL_STATS)
display(RATIO_AUDIT)


Enzyme | fold 1 | MSCMF | RandomOver


NameError: name 'valid_saved_pool' is not defined

# Step 10 — Augmentation audit

## 9. Reconstruct and validate every saved ratio

In [11]:
def assemble_augmented_training_set(
    family,
    outer_fold,
    backbone,
    method,
    ratio_name,
):
    source = load_training_representation(
        family,
        outer_fold,
        backbone,
    )

    X = source["X"]
    y = source["y"]

    out_dir = augmentation_dir(
        family,
        outer_fold,
        backbone,
        method,
    )

    synthetic = np.load(
        out_dir
        / "synthetic_pool.npy"
    ).astype(np.float32)

    plan = np.load(
        out_dir
        / f"ratio_{ratio_name}_plan.npz"
    )

    real_indices = plan[
        "real_indices"
    ].astype(np.int64)

    synthetic_indices = plan[
        "synthetic_indices"
    ].astype(np.int64)

    X_real = X[
        real_indices
    ].astype(np.float32)
    y_real = y[
        real_indices
    ].astype(np.int8)

    if len(synthetic_indices):
        X_syn = synthetic[
            synthetic_indices
        ].astype(np.float32)

        y_syn = np.ones(
            len(synthetic_indices),
            dtype=np.int8,
        )

        X_final = np.vstack([
            X_real,
            X_syn,
        ])
        y_final = np.concatenate([
            y_real,
            y_syn,
        ])
    else:
        X_final = X_real
        y_final = y_real

    return (
        X_final,
        y_final,
        real_indices,
        synthetic_indices,
        source,
    )


RECONSTRUCTION_AUDIT_ROWS = []

for family, cfg in FAMILIES.items():

    for outer_fold in range(
        1,
        N_OUTER_FOLDS + 1,
    ):

        for backbone, method in (
            METHOD_BACKBONE_CONFIGS
        ):

            for ratio_name, (
                neg_units,
                pos_units,
            ) in TARGET_RATIOS.items():

                (
                    X_final,
                    y_final,
                    real_indices,
                    synthetic_indices,
                    source,
                ) = (
                    assemble_augmented_training_set(
                        family,
                        outer_fold,
                        backbone,
                        method,
                        ratio_name,
                    )
                )

                n0 = int(
                    (y_final == 0).sum()
                )
                n1 = int(
                    (y_final == 1).sum()
                )

                exact_ratio_pass = (
                    n0 * pos_units
                    == n1 * neg_units
                )

                shape_pass = (
                    X_final.ndim == 2
                    and X_final.shape[1]
                    == FEATURE_DIM
                    and len(X_final)
                    == len(y_final)
                )

                finite_pass = bool(
                    np.isfinite(
                        X_final
                    ).all()
                )

                binary_labels_pass = (
                    set(
                        np.unique(
                            y_final
                        )
                    ).issubset({0, 1})
                )

                train_roles = source[
                    "train_roles"
                ]

                selected_pair_ids = set(
                    train_roles.iloc[
                        real_indices
                    ]["pair_id"]
                )

                roles_all = source[
                    "roles_all"
                ]

                val_ids = set(
                    roles_all.loc[
                        roles_all[
                            "role"
                        ].eq(
                            "validation"
                        ),
                        "pair_id",
                    ]
                )

                test_ids = set(
                    roles_all.loc[
                        roles_all[
                            "role"
                        ].eq("test"),
                        "pair_id",
                    ]
                )

                leakage_pass = (
                    selected_pair_ids
                    .isdisjoint(
                        val_ids
                    )
                    and selected_pair_ids
                    .isdisjoint(
                        test_ids
                    )
                )

                final_pass = (
                    exact_ratio_pass
                    and shape_pass
                    and finite_pass
                    and binary_labels_pass
                    and leakage_pass
                )

                assert final_pass

                RECONSTRUCTION_AUDIT_ROWS.append({
                    "family": cfg["display"],
                    "outer_fold": outer_fold,
                    "backbone": backbone,
                    "method": method,
                    "ratio": ratio_name,
                    "n_non_interaction": n0,
                    "n_interaction": n1,
                    "n_total": len(y_final),
                    "n_real": len(
                        real_indices
                    ),
                    "n_synthetic": len(
                        synthetic_indices
                    ),
                    "exact_ratio_pass": (
                        exact_ratio_pass
                    ),
                    "shape_pass": (
                        shape_pass
                    ),
                    "finite_values_pass": (
                        finite_pass
                    ),
                    "binary_labels_pass": (
                        binary_labels_pass
                    ),
                    "train_only_pass": (
                        leakage_pass
                    ),
                    "final_pass": (
                        final_pass
                    ),
                })

RECONSTRUCTION_AUDIT = pd.DataFrame(
    RECONSTRUCTION_AUDIT_ROWS
)

RECONSTRUCTION_AUDIT.to_excel(
    STATS_DIR
    / "augmentation_reconstruction_audit.xlsx",
    index=False,
    sheet_name="Reconstruction Audit",
)

assert RECONSTRUCTION_AUDIT[
    "final_pass"
].all()

display(RECONSTRUCTION_AUDIT)

,family,outer_fold,backbone,method,ratio,n_non_interaction,n_interaction,n_total,n_real,n_synthetic,exact_ratio_pass,shape_pass,finite_values_pass,binary_labels_pass,train_only_pass,final_pass
0,Enzyme,1,MSCMF,RandomOver,50_50,187235,187235,374470,189107,185363,True,True,True,True,True,True
1,Enzyme,1,MSCMF,RandomOver,80_20,187232,46808,234040,189104,44936,True,True,True,True,True,True
2,Enzyme,1,MSCMF,RandomOver,85_15,187221,33039,220260,189093,31167,True,True,True,True,True,True
3,Enzyme,1,MSCMF,RandomOver,90_10,187227,20803,208030,189099,18931,True,True,True,True,True,True
4,Enzyme,1,MSCMF,RandomUnder,50_50,1872,1872,3744,3744,0,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,Nuclear Receptor,5,LMF,WGANGP,90_10,837,93,930,895,35,True,True,True,True,True,True
1436,Nuclear Receptor,5,MSCMF,LatentDiffusion,50_50,841,841,1682,899,783,True,True,True,True,True,True
1437,Nuclear Receptor,5,MSCMF,LatentDiffusion,80_20,840,210,1050,898,152,True,True,True,True,True,True
1438,Nuclear Receptor,5,MSCMF,LatentDiffusion,85_15,833,147,980,891,89,True,True,True,True,True,True


## 11. Method protocol table

In [12]:
PROTOCOL_ROWS = []

for backbone, method in METHOD_BACKBONE_CONFIGS:
    metadata = method_metadata(
        backbone,
        method,
    )

    PROTOCOL_ROWS.append({
        "backbone": backbone,
        "method": method,
        "applicability": (
            "MSCMF only"
            if (
                method
                in CLASSICAL_MSCMF_METHODS
                or method == "LatentDiffusion"
            )
            else "MSCMF, NNMF, LMF"
        ),
        "parameters_json": json.dumps(
            metadata["parameters"],
            sort_keys=True,
        ),
        "train_only": True,
        "target_ratios": (
            "50/50; 80/20; 85/15; 90/10"
        ),
        "feature_dimension": FEATURE_DIM,
        "implementation_version": (
            IMPLEMENTATION_VERSION
        ),
    })

METHOD_PROTOCOL = pd.DataFrame(
    PROTOCOL_ROWS
)

METHOD_PROTOCOL.to_excel(
    STATS_DIR
    / "augmentation_method_protocol.xlsx",
    index=False,
    sheet_name="Method Protocol",
)

display(METHOD_PROTOCOL)

,backbone,method,applicability,parameters_json,train_only,target_ratios,feature_dimension,implementation_version
0,MSCMF,RandomOver,MSCMF only,"{""random_state"": 42, ""replace"": true}",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
1,MSCMF,RandomUnder,MSCMF only,"{""random_state"": 42, ""replace"": false}",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
2,MSCMF,TomekLinks,MSCMF only,{},True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
3,MSCMF,CNNTomekLinks,MSCMF only,"{""cnn_n_neighbors"": 1, ""random_state"": 42, ""st...",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
4,MSCMF,SMOTE,MSCMF only,"{""random_state"": 42, ""standard_scaler"": true}",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
5,MSCMF,SMOTETomekLinks,MSCMF only,"{""random_state"": 42, ""standard_scaler"": true}",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
6,MSCMF,BasicSMOTE,MSCMF only,"{""random_state"": 42, ""standard_scaler"": false}",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
7,MSCMF,ADASYN,MSCMF only,"{""random_state"": 42}",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
8,MSCMF,KMeansSMOTE,MSCMF only,"{""cluster_balance_threshold"": 0.01, ""n_cluster...",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...
9,MSCMF,LatentSampling,MSCMF only,"{""covariance_type"": ""full"", ""model"": ""Gaussian...",True,50/50; 80/20; 85/15; 90/10,100,augmentation_v2_train_only_14_methods_with_dif...


## 12. Consolidated Step 09-10 workbook

In [ ]:
from pathlib import Path
import os

import pandas as pd


# ============================================================
# 1. Project paths
# ============================================================

PROJECT_ROOT = Path.cwd()

STATS_DIR = (
    PROJECT_ROOT
    / "Stats"
)

STATS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Individual source workbooks
# ============================================================

input_audit_file = (
    STATS_DIR
    / "augmentation_input_audit.xlsx"
)

pool_stats_file = (
    STATS_DIR
    / "augmentation_pool_statistics.xlsx"
)

ratio_audit_file = (
    STATS_DIR
    / "augmentation_ratio_audit.xlsx"
)

reconstruction_audit_file = (
    STATS_DIR
    / "augmentation_reconstruction_audit.xlsx"
)

method_protocol_file = (
    STATS_DIR
    / "augmentation_method_protocol.xlsx"
)


# ============================================================
# 3. Verify that ALL source workbooks exist
#
# ============================================================

required_files = {

    "Input Audit":
        input_audit_file,

    "Pool Statistics":
        pool_stats_file,

    "Ratio Audit":
        ratio_audit_file,

    "Reconstruction Audit":
        reconstruction_audit_file,

    "Method Protocol":
        method_protocol_file,
}


missing_files = [

    str(path)

    for path
    in required_files.values()

    if not path.exists()
]


if missing_files:

    print(
        "\nMissing source workbooks:"
    )

    for path in missing_files:

        print(
            " -",
            path
        )

    raise FileNotFoundError(
        "The consolidated workbook was NOT created "
        "because one or more source Stats files are missing."
    )


print(
    "PASS - All source workbooks exist."
)


# ============================================================
# 4. Load all source DataFrames BEFORE creating output
# ============================================================

INPUT_AUDIT = pd.read_excel(
    input_audit_file,
    sheet_name="Input Audit",
)

POOL_STATS = pd.read_excel(
    pool_stats_file,
    sheet_name="Pool Statistics",
)

RATIO_AUDIT = pd.read_excel(
    ratio_audit_file,
    sheet_name="Ratio Audit",
)

RECONSTRUCTION_AUDIT = pd.read_excel(
    reconstruction_audit_file,
    sheet_name="Reconstruction Audit",
)

METHOD_PROTOCOL = pd.read_excel(
    method_protocol_file,
    sheet_name="Method Protocol",
)


# ============================================================
# 5. Basic source validation
# ============================================================

dataframes = {

    "Input Audit":
        INPUT_AUDIT,

    "Pool Statistics":
        POOL_STATS,

    "Ratio Audit":
        RATIO_AUDIT,

    "Reconstruction Audit":
        RECONSTRUCTION_AUDIT,

    "Method Protocol":
        METHOD_PROTOCOL,
}


for sheet_name, df in dataframes.items():

    if not isinstance(
        df,
        pd.DataFrame,
    ):

        raise TypeError(
            f"{sheet_name} did not load "
            f"as a pandas DataFrame."
        )

    if len(df.columns) == 0:

        raise ValueError(
            f"{sheet_name} contains no columns."
        )

    print(
        f"PASS - "
        f"{sheet_name}: "
        f"{len(df):,} rows x "
        f"{len(df.columns)} columns"
    )


# ============================================================
# 6. Output filenames
# ============================================================

consolidated_file = (
    STATS_DIR
    / "augmentation_9_10_statistics.xlsx"
)

temporary_file = (
    STATS_DIR
    / "augmentation_9_10_statistics_TEMP.xlsx"
)


# ============================================================
# 7. Remove stale temporary file only
# ============================================================

if temporary_file.exists():

    temporary_file.unlink()


# ============================================================
# 8. Write TEMPORARY consolidated workbook
# ============================================================

with pd.ExcelWriter(
    temporary_file,
    engine="openpyxl",
) as writer:

    INPUT_AUDIT.to_excel(
        writer,
        sheet_name="Input Audit",
        index=False,
    )

    POOL_STATS.to_excel(
        writer,
        sheet_name="Pool Statistics",
        index=False,
    )

    RATIO_AUDIT.to_excel(
        writer,
        sheet_name="Ratio Audit",
        index=False,
    )

    RECONSTRUCTION_AUDIT.to_excel(
        writer,
        sheet_name="Reconstruction Audit",
        index=False,
    )

    METHOD_PROTOCOL.to_excel(
        writer,
        sheet_name="Method Protocol",
        index=False,
    )


print(
    "\nTemporary workbook created:",
    temporary_file
)


# ============================================================
# 9. Validate temporary workbook
# ============================================================

expected_sheets = [

    "Input Audit",
    "Pool Statistics",
    "Ratio Audit",
    "Reconstruction Audit",
    "Method Protocol",
]


with pd.ExcelFile(
    temporary_file,
    engine="openpyxl",
) as workbook:

    actual_sheets = (
        workbook.sheet_names
    )


if actual_sheets != expected_sheets:

    raise AssertionError(
        "Unexpected workbook sheets.\n"
        f"Expected: {expected_sheets}\n"
        f"Found:    {actual_sheets}"
    )


# ============================================================
# 10. Re-open every sheet from temporary workbook
#
# ============================================================

for sheet_name in expected_sheets:

    test_df = pd.read_excel(
        temporary_file,
        sheet_name=sheet_name,
        engine="openpyxl",
    )

    if (
        len(
            test_df.columns
        )
        == 0
    ):

        raise ValueError(
            f"Validation failed: "
            f"{sheet_name} has no columns."
        )

    print(
        f"PASS - Temporary sheet readable: "
        f"{sheet_name} "
        f"({len(test_df):,} rows)"
    )


# ============================================================
# 11. Replace corrupted/old final
# ============================================================

os.replace(
    temporary_file,
    consolidated_file,
)


# ============================================================
# 12. Final validation
# ============================================================

if not consolidated_file.exists():

    raise FileNotFoundError(
        "Final consolidated workbook "
        "was not created."
    )


with pd.ExcelFile(
    consolidated_file,
    engine="openpyxl",
) as final_workbook:

    final_sheets = (
        final_workbook.sheet_names
    )


assert (
    final_sheets
    == expected_sheets
)


# ============================================================
# 13. Final report
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PASS - Consolidated Step 09-10 "
    "workbook created successfully."
)

print(
    "File:",
    consolidated_file
)

print(
    "Sheets:",
    final_sheets
)

print(
    "=" * 80
)

PASS - All source workbooks exist.
PASS - Input Audit: 20 rows x 9 columns
PASS - Pool Statistics: 360 rows x 15 columns
PASS - Ratio Audit: 1,440 rows x 18 columns
PASS - Reconstruction Audit: 1,440 rows x 16 columns
PASS - Method Protocol: 18 rows x 8 columns

Temporary workbook created: c:\Users\riskf\OneDrive\A-DTI2026\Stats\augmentation_9_10_statistics_TEMP.xlsx
PASS - Temporary sheet readable: Input Audit (20 rows)
PASS - Temporary sheet readable: Pool Statistics (360 rows)
PASS - Temporary sheet readable: Ratio Audit (1,440 rows)
PASS - Temporary sheet readable: Reconstruction Audit (1,440 rows)
PASS - Temporary sheet readable: Method Protocol (18 rows)

PASS - Consolidated Step 09-10 workbook created successfully.
File: c:\Users\riskf\OneDrive\A-DTI2026\Stats\augmentation_9_10_statistics.xlsx
Sheets: ['Input Audit', 'Pool Statistics', 'Ratio Audit', 'Reconstruction Audit', 'Method Protocol']


## 13. Save Step 09-10 manifest

In [17]:
manifest = {
    "implementation_version": (
        IMPLEMENTATION_VERSION
    ),
    "steps": [
        "09 - augment inner-training observations only",
        "10 - audit augmentation",
    ],
    "families": list(
        FAMILIES.keys()
    ),
    "outer_folds": (
        N_OUTER_FOLDS
    ),
    "unique_augmentation_algorithms": 14,
    "method_backbone_configurations": [
        {
            "backbone": backbone,
            "method": method,
        }
        for backbone, method
        in METHOD_BACKBONE_CONFIGS
    ],
    "target_ratios": {
        key: {
            "non_interaction_units": value[0],
            "interaction_units": value[1],
        }
        for key, value
        in TARGET_RATIOS.items()
    },
    "leakage_rule": (
        "augmentation receives role=train observations only"
    ),
    "validation_used": False,
    "outer_test_used": False,
    "storage": (
        "compact real-index + synthetic-pool + ratio-plan format"
    ),
    "stats_workbook": str(
        consolidated_file.relative_to(
            PROJECT_ROOT
        )
    ),
}

manifest_file = (
    SPLIT_DIR
    / "augmentation_9_10_manifest.json"
)

manifest_file.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Manifest saved to:",
    manifest_file
)

Manifest saved to: c:\Users\riskf\OneDrive\A-DTI2026\data\split\augmentation_9_10_manifest.json


## 14. Final integrity check

Expected number of ratio-level audited configurations:

\[
4\ \text{families}
\times
5\ \text{folds}
\times
18\ \text{method-backbone configurations}
\times
4\ \text{ratios}
=
1440.
\]

The next experimental step is Step 11: train the fixed DNN on each reconstructed inner-training dataset while leaving validation and outer-test observations untouched.

In [18]:
expected_ratio_rows = (
    len(FAMILIES)
    * N_OUTER_FOLDS
    * len(
        METHOD_BACKBONE_CONFIGS
    )
    * len(TARGET_RATIOS)
)

assert expected_ratio_rows == 1440
assert len(RATIO_AUDIT) == expected_ratio_rows
assert (
    len(RECONSTRUCTION_AUDIT)
    == expected_ratio_rows
)

assert RATIO_AUDIT[
    "pass"
].all()

assert RECONSTRUCTION_AUDIT[
    "final_pass"
].all()

required_stats_files = [
    "augmentation_input_audit.xlsx",
    "augmentation_pool_statistics.xlsx",
    "augmentation_ratio_audit.xlsx",
    "augmentation_reconstruction_audit.xlsx",
    "augmentation_method_protocol.xlsx",
    "augmentation_9_10_statistics.xlsx",
]

for filename in required_stats_files:
    assert (
        STATS_DIR / filename
    ).exists(), (
        f"Missing Stats file: {filename}"
    )

assert manifest_file.exists()

print(
    "PASS - Step 09 augmentation "
    "uses inner-training observations only."
)
print(
    "PASS - Step 10 augmentation "
    "audit completed."
)
print(
    "PASS - 14 augmentation algorithms, "
    "18 method-backbone configurations, "
    "4 ratios."
)
print(
    "\nNotebook 9-10 completed successfully."
)
print(
    "Next: Step 11 - fixed DNN training."
)

PASS - Step 09 augmentation uses inner-training observations only.
PASS - Step 10 augmentation audit completed.
PASS - 14 augmentation algorithms, 18 method-backbone configurations, 4 ratios.

Notebook 9-10 completed successfully.
Next: Step 11 - fixed DNN training.
